In [1]:
import numpy as np
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta, date, time
import random

In [ ]:
fake = Faker('es_ES')
np.random.seed(42)
random.seed(42)

# PARÁMETROS GENERALES
FECHA_INICIO = date(2024, 1, 1)
FECHA_FIN   = date(2025, 12, 31)

N_PACIENTES = 1500          # pacientes totales aprox
N_FISIOTERAPEUTAS = 5          # ya definidos en MySQL
N_TRATAMIENTOS = 6          # ya definidos en MySQL
SALAS = ['Sala 1', 'Sala 2', 'Sala 3', 'Sala entrenamiento']

# Porcentajes de origen de pacientes
ORIGEN = ['Privado', 'Recomendación', 'Redes sociales', 'Otro']
PROB_ORIGEN = [0.30, 0.35, 0.25, 0.10]

# Distribución nº de citas por paciente
N_CITAS = [1, 4, 6, 10, 20]
PROB_CITAS   = [0.4, 0.3, 0.15, 0.1, 0.05]

# Probabilidades de estado de la cita
ESTADO = ['Realizada', 'Cancelada', 'No viene']
PROB_ESTADO = [0.88, 0.05, 0.07]

# Probabilidades de formato de pago
PAGO = ['Sesión', 'Bono', 'Tarjeta regalo']
PROB_PAGO = [0.6, 0.35, 0.05]

# Probabilidades de tipo de pago
TIPO_PAGO = ['Efectivo', 'Tarjeta']
PROB_TIPO_PAGO = [0.2, 0.8]

# Precio base por id_tratamiento (iguales que en SQL)
PRECIOS_TRATAMIENTOS = {
    1: 50.0,  # Fisioterapia
    2: 60.0,  # Suelo pélvico
    3: 55.0,  # Diatermia
    4: 65.0,  # Ejercicio terapéutico
    5: 50.0,  # Fisioterapia infantil
    6: 55.0   # Drenaje linfático manual
}

# Para bonos
SESIONES_BONO = 5
DESCUENTO_BONO = 0.8   # 20% descuento vs precio suelto

In [10]:
# Funciones para generar los datos
# -----------------------

def fecha_aleatoria(start, end):
    # Si end es igual o anterior a start, devolvemos start
    if end <= start:
        return start

    delta = end - start
    dias = delta.days
    random_days = np.random.randint(0, dias + 1)
    return start + timedelta(days = int(random_days))

def dia_hora_aleatoria(fecha):
    """
    Devuelve una hora de cita realista:
    L-V, entre 9-14 y 16-21, con preferencia por tardes.
    """
    # evitar fines de semana
    while fecha.weekday() >= 5:
        fecha += timedelta(days = 1)

    # tramos horarios (tardes con más probabilidad)
    slots = [
        (9,  14, 0.4),  # mañana
        (16, 21, 0.6)   # tarde
    ]
    r = np.random.rand()
    if r < slots[0][2]:
        start_h, end_h = slots[0][0], slots[0][1]
    else:
        start_h, end_h = slots[1][0], slots[1][1]

    hour = np.random.randint(start_h, end_h)
    return date(fecha.year, fecha.month, fecha.day), time(hour)

def choose_citas_count():
    """Elige cuántas citas tendrá un paciente (1, 2-4, 5-8 aprox)."""
    bucket = np.random.choice(N_CITAS, p = PROB_CITAS)
    if bucket == 1:
        return 1
    elif bucket == 4:
        return np.random.randint(2, 4)   # 2-4
    elif bucket == 6:
        return np.random.randint(5, 8)   # 5-8
    elif bucket == 10:
        return np.random.randint(9, 12)   # 9-12
    else:
        return np.random.randint(13, 25)   # 13-25


def generate_patients(n_pacientes):
    pacientes = []
    for pid in range(1, n_pacientes + 1):
        gender = np.random.choice(['F', 'M', 'O'], p = [0.5, 0.48, 0.02])
        profile = fake.simple_profile(sex = 'F' if gender == 'F' else 'M')
        first_name = profile['name'].split()[0]
        last_name  = ' '.join(profile['name'].split()[1:]) if len(profile['name'].split()) > 1 else fake.last_name()

        # edad entre 18 y 80
        age = np.random.randint(18, 81)
        birth_year = datetime.now().year - age
        birth_date = date(birth_year, np.random.randint(1, 13), np.random.randint(1, 28))

        fecha_registro = fecha_aleatoria(FECHA_INICIO, FECHA_FIN)
        origen = np.random.choice(ORIGEN, p = PROB_ORIGEN)

        pacientes.append({
            'id_paciente': pid,
            'nombre': first_name,
            'apellidos': last_name,
            'genero': gender,
            'fecha_nacimiento': birth_date,
            'origen': origen,
            'fecha_registro': fecha_registro
        })
    return pd.DataFrame(pacientes)

def generate_bonos_for_patients(df_pacientes):
    """
    Genera bonos para ~15-20% de los pacientes.
    Bonos solo para treatment_id = 1 (Fisioterapia).
    """
    bonos = []
    bono_id = 1
    # Seleccionamos un subconjunto de pacientes
    pacientes_con_bono = df_pacientes.sample(frac = 0.18, random_state = 42)

    for _, row in pacientes_con_bono.iterrows():
        n_bonos_paciente = np.random.choice([1, 2], p = [0.8, 0.2])

        # cada paciente puede tener 1 o 2 bonos a lo largo del tiempo
        for _ in range(n_bonos_paciente):
            total_sesiones = SESIONES_BONO
            base_price = PRECIOS_TRATAMIENTOS[1]
            precio_por_sesion = round(base_price * DESCUENTO_BONO, 2)
            cantidad_total = round(precio_por_sesion * total_sesiones, 2)
            
            start_date = max(FECHA_INICIO, row['fecha_registro'])
            end_date = FECHA_FIN - timedelta(days = 60)

            if start_date < end_date:  # Solo si hay rango válido
                fecha_compra = fecha_aleatoria(start_date, end_date)
            else:
                fecha_compra = start_date

            # fecha_compra = fecha_aleatoria(
            #     max(FECHA_INICIO, row['fecha_registro']),
            #     FECHA_FIN - timedelta(days = 60)
            # )
            fecha_caducidad = fecha_compra + timedelta(days = 180)  # 6 meses

            bonos.append({
                'id_bono': bono_id,
                'id_paciente': row['id_paciente'],
                'id_tratamiento': 1,  # Sesión de fisio normal
                'total_sesiones': total_sesiones,
                'sesiones_consumidas': 0,       # se actualizará al generar citas
                'precio': cantidad_total,
                'precio_sesion': precio_por_sesion,
                'fecha_compra': fecha_compra,
                'fecha_caducidad': fecha_caducidad,
                'estado': 'Activo'        # luego ajustaremos a Expirado/Agotado
            })
            bono_id += 1

    return pd.DataFrame(bonos)

def generate_appointments_and_update_bonos(df_pacientes, df_bonos):
    citas = []
    id_cita = 1

    # Índice rápido de bonos por paciente
    bonos_por_paciente = {pid: [] for pid in df_pacientes['id_paciente']}
    for idx, row in df_bonos.iterrows():
        bonos_por_paciente[row['id_paciente']].append(row)

    for _, pacientes in df_pacientes.iterrows():
        n_citas = choose_citas_count()

        # primera cita cerca de la fecha de registro
        first_date = fecha_aleatoria(
            max(FECHA_INICIO, pacientes['fecha_registro']),
            FECHA_FIN - timedelta(days = 1)
        )
        # aseguramos que no sea fin de semana
        current_date = first_date

        # si el paciente tiene bonos, los usaremos en parte de las citas
        bonos_pacientes = bonos_por_paciente.get(pacientes['id_paciente'], [])

        for i in range(n_citas):
            # definimos tipo de pago según probabilidades, pero si hay bono disponible,
            # aumentamos la probabilidad de 'Bono'
            base_probs = PROB_PAGO.copy()
            if bonos_pacientes:
                # aumentamos algo la probabilidad de que sea bono
                base_probs = [0.25, 0.70, 0.05]  # algo más de 'Bono'

            pago = np.random.choice(PAGO, p = base_probs)

            # elegimos tratamiento: si pago con bono, treatment_id = 1
            if pago == 'Bono' and bonos_pacientes:
                treatment_id = 1
            else:
                # distribución simple: más probabilidad de seguimiento
                treatment_id = np.random.choice(
                    [2, 3, 4, 5, 6],
                    p = [0.25, 0.50, 0.10, 0.10, 0.05]
                )

            # generar fecha/hora realista
            appt_datetime = dia_hora_aleatoria(current_date)[0]
            hora = dia_hora_aleatoria(current_date)[1]

            status = np.random.choice(ESTADO, p = PROB_ESTADO)

            # precio base
            price = PRECIOS_TRATAMIENTOS[treatment_id]
            # # ligera variación +/- 5%
            # variation = np.random.uniform(0.95, 1.05)
            # amount = round(base_price * variation, 2)

            # si se paga con bono, buscamos un bono con sesiones disponibles
            if pago == 'Bono' and bonos_pacientes:
                usable_bonos = [b for b in bonos_pacientes if b['sesiones_consumidas'] < b['total_sesiones']]
                if usable_bonos:
                    bono = random.choice(usable_bonos)
                    # usar precio por sesión del bono
                    price = bono['precio_sesion']

                    # si la cita se realiza, consumimos una sesión
                    if status == 'Realizada':
                        bono['sesiones_consumidas'] += 1

            tipo_pago = np.random.choice(TIPO_PAGO, p = PROB_TIPO_PAGO)

            # # tipo de pago 'Seguro' o 'Mutua' podría facturar un poco menos
            # if payment_type in ['Seguro', 'Mutua']:
            #     amount = round(amount * np.random.uniform(0.7, 0.9), 2)

            therapist_id = np.random.randint(1, N_FISIOTERAPEUTAS + 1)
            if treatment_id == 4:
                room = 'Sala entrenamiento'
            else:
                salas_1 = SALAS.copy()
                salas_1.remove('Sala entrenamiento')
                room = random.choice(salas_1)

            fecha_creacion = appt_datetime - timedelta(days = np.random.randint(1, 15))

            citas.append({
                'id_cita': id_cita,
                'id_paciente': pacientes['id_paciente'],
                'id_fisioterapeuta': therapist_id,
                'id_tratamiento': treatment_id,
                'fecha_cita': appt_datetime,
                'hora_cita': hora,
                'estado': status,
                'dinero_pagado': pago,
                'sala': room,
                'tipo_pago': tipo_pago,
                'fecha_creación': fecha_creacion
            })
            id_cita += 1

            # siguiente cita 3-14 días después
            current_date = current_date + timedelta(days = np.random.randint(3, 15))

    # actualizar estado de bonos (Agotado / Expirado / Activo)
    for idx, row in df_bonos.iterrows():
        used = row['sesiones_consumidas']
        total = row['total_sesiones']
        if used >= total:
            df_bonos.at[idx, 'estado'] = 'Agotado'
        else:
            # si la fecha actual es posterior a expiry, lo marcamos Expirado
            if row['fecha_caducidad'] < FECHA_FIN:
                df_bonos.at[idx, 'estado'] = 'Expirado'
            else:
                df_bonos.at[idx, 'estado'] = 'Activo'

    return pd.DataFrame(citas), df_bonos

In [ ]:
# GENERACIÓN DE DATOS
df_pacientes = generate_patients(N_PACIENTES)
df_bonos = generate_bonos_for_patients(df_pacientes)
df_citas, df_bonos = generate_appointments_and_update_bonos(df_pacientes, df_bonos)

# Ordenamos por fecha para que quede más limpio
appointments_df = df_citas.sort_values(by = 'fecha_cita').reset_index(drop = True)

# EXPORTAR A CSV
df_pacientes.to_csv('pacientes.csv', index = False)
df_citas.to_csv('citas.csv', index = False)
df_bonos.to_csv('bonos.csv', index = False)

print('CSV generados: pacientes.csv, citas.csv, bonos.csv')

CSV generados: pacientes.csv, citas.csv, bonos.csv


In [ ]:
# # generate_clinic_data.py
# # Solo necesita: numpy, pandas, Faker
# import numpy as np
# import pandas as pd
# from faker import Faker
# from datetime import datetime, timedelta
# import random

# if __name__ == "__main__":
#     # generar datos
#     patients_df = generate_patients(700)
#     bonos_df = generate_bonos_for_patients(patients_df)
#     appointments_df, bonos_df = generate_appointments_and_update_bonos(patients_df, bonos_df)
    
#     # ordenar citas por fecha
#     appointments_df = appointments_df.sort_values(by='appointment_datetime').reset_index(drop=True)
    
#     # exportar
#     patients_df.to_csv('patients.csv', index=False)
#     appointments_df.to_csv('appointments.csv', index=False)
#     bonos_df.to_csv('bonos.csv', index=False)
    
#     print("✅ CSV generados: patients.csv, appointments.csv, bonos.csv")
#     print(f"📊 Total pacientes: {len(patients_df)}")
#     print(f"📅 Total citas: {len(appointments_df)}")
#     print(f"🎟️ Total bonos: {len(bonos_df)}")
